# PathSnap | Prepare the evaluator trips

Place this notebook beside the teacher’s `evaluator_test/` directory. With NumPy, SciPy, pandas and a Jupyter Python kernel installed, select **Restart Kernel and Run All Cells**. Each source trip needs its headerless track (longitude, latitude, timestamp), nodes (longitude, latitude) and arcs (start and end node indices). This notebook contains the same coordinate projection, candidate generator and 11 feature definitions used in training.

**Output:** `cleaned_data/evaluator_test/<trip>/` contains canonical numeric track and graph files for live inference plus `candidates.csv` for inspecting the feature values. The partition inventory summarises the data-quality checks. Reference routes stay in the original directory for optional scoring by the server after prediction. Preparation requires neither labels nor a trained estimator.

In [1]:
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display

SOURCE = Path("evaluator_test")
OUTPUT = Path("cleaned_data/evaluator_test")
DEFAULT_K = 8
FEATURES = ("log_distance", "distance_above_best", "along_fraction", "log_length",
            "incoming_alignment", "outgoing_alignment", "window_alignment", "incoming_speed",
            "outgoing_speed", "incoming_overshoot", "outgoing_overshoot")
trips = sorted(p for p in SOURCE.iterdir() if p.is_dir())
if not trips:
    raise ValueError(f"No trip directories found under {SOURCE}")
print(f"Found {len(trips)} evaluator trips.")

Found 10 evaluator trips.


## Validate and project each track

Keep all finite observations in their original order and check graph indices and timestamp ordering. A local metric projection supplies distances and direction vectors. Input row positions identify nodes and road arcs, so cleaning preserves duplicates and original row positions. Time gaps are recorded for the decoder.

In [2]:
@dataclass
class NetworkTrack:
    track: np.ndarray
    nodes: np.ndarray
    arcs: np.ndarray
    gps: np.ndarray
    start: np.ndarray
    vector: np.ndarray
    length: np.ndarray


def read_trip(directory):
    def load(ext, dtype):
        (path,) = directory.glob("*." + ext)
        return np.loadtxt(path, dtype=dtype, ndmin=2)

    track, nodes, arcs = load("track", float), load("nodes", float), load("arcs", int)
    if track.shape[1] != 3 or nodes.shape[1] != 2 or arcs.shape[1] != 2:
        raise ValueError("Invalid track or network columns")
    if (
        not len(track)
        or not len(arcs)
        or not np.isfinite(track).all()
        or not np.isfinite(nodes).all()
    ):
        raise ValueError("Missing or nonfinite track/network")
    if arcs.min() < 0 or arcs.max() >= len(nodes) or np.any(np.diff(track[:, 2]) < 0):
        raise ValueError("Invalid node index or unordered track")
    origin = nodes.mean(axis=0)
    scale = np.pi / 180 * 6371008.8 * np.array([np.cos(np.deg2rad(origin[1])), 1.0])
    xy, gps = (nodes - origin) * scale, (track[:, :2] - origin) * scale
    start = xy[arcs[:, 0]]
    vector = xy[arcs[:, 1]] - start
    return NetworkTrack(track, nodes, arcs, gps, start, vector, np.linalg.norm(vector, axis=1))


def projection(net, points, ids):
    offset = points - net.start[ids]
    along = (offset * net.vector[ids]).sum(axis=-1) / np.maximum(net.length[ids] ** 2, 1e-15)
    fraction = np.clip(along, 0, 1)
    distance = np.linalg.norm(offset - fraction[..., None] * net.vector[ids], axis=-1)
    return distance, fraction

## Calculate candidate features

Rank positive-length directed arcs by exact distance from the observation to the segment. Generate up to eight candidates, retaining opposite directions. Compute distance, position along the road, length, motion alignment, speed and endpoint overshoot with the training transformations. The CSV records candidate rank and graph indices for tracing a row back to the input.

In [3]:
def candidates(net, k, spacing=40.0):
    """Midpoints of subsegments accelerate exact nearest-segment queries."""
    valid = np.flatnonzero(net.length > 0)
    if not len(valid):
        raise ValueError("No positive-length arcs")
    k = min(k, len(valid))
    pieces = np.maximum(1, np.ceil(net.length[valid] / spacing).astype(int))
    ids = np.repeat(valid, pieces)
    local = np.arange(len(ids)) - np.repeat(np.cumsum(pieces) - pieces, pieces)
    fraction = (local + 0.5) / np.repeat(pieces, pieces)
    tree = cKDTree(net.start[ids] + fraction[:, None] * net.vector[ids])
    midpoints = cKDTree(net.start[valid] + 0.5 * net.vector[valid])
    _, upper_ids = midpoints.query(net.gps, k=list(range(1, k + 1)))
    upper = projection(net, net.gps[:, None], valid[upper_ids])[0].max(axis=1)
    # Triangle inequality gives a conservative bound for a witness midpoint.
    hits = tree.query_ball_point(net.gps, upper + spacing / 2 + 1e-7)
    counts = np.fromiter(map(len, hits), int)
    obs = np.repeat(np.arange(len(hits)), counts)
    pair = np.unique(obs * len(net.arcs) + ids[np.concatenate(hits).astype(int)])
    obs, arcs = np.divmod(pair, len(net.arcs))
    distance, _ = projection(net, net.gps[obs], arcs)
    # Geometric secondary keys keep boundary ties independent of input row order.
    order = np.lexsort(
        (
            net.vector[arcs, 1],
            net.vector[arcs, 0],
            net.start[arcs, 1],
            net.start[arcs, 0],
            distance,
            obs,
        )
    )
    counts = np.bincount(obs)
    rank = np.arange(len(pair)) - np.repeat(np.cumsum(counts) - counts, counts)
    return arcs[order[rank < k]].reshape(-1, k)


def features(net, k=DEFAULT_K):
    ids = candidates(net, k)
    distance, fraction = projection(net, net.gps[:, None], ids)
    length = net.length[ids]
    unit = net.vector[ids] / length[:, :, None]
    idx = np.arange(len(net.track))

    def motion(a, b):
        delta = net.gps[b] - net.gps[a]
        norm = np.linalg.norm(delta, axis=1)
        cosine = (unit * delta[:, None, :]).sum(axis=2) / np.maximum(norm[:, None], 1e-12)
        dt = net.track[b, 2] - net.track[a, 2]
        speed = np.divide(norm, dt, out=np.zeros_like(norm), where=dt > 0)
        return cosine, np.broadcast_to(np.log1p(speed[:, None]), ids.shape), norm[:, None]

    cin, vin, din = motion(np.maximum(0, idx - 1), idx)
    cout, vout, dout = motion(idx, np.minimum(idx + 1, idx[-1]))
    wide, _, _ = motion(np.maximum(0, idx - 3), np.minimum(idx + 3, idx[-1]))
    x = np.stack(
        (
            np.log1p(distance),
            np.log1p(distance - distance.min(axis=1, keepdims=True)),
            fraction,
            np.log1p(length),
            cin,
            cout,
            wide,
            vin,
            vout,
            np.log1p(np.maximum(0, din - fraction * length)),
            np.log1p(np.maximum(0, dout - (1 - fraction) * length)),
        ),
        axis=-1,
    )
    return ids, x.astype("float32")

## Write the demo inputs and inspect the inventory

Normalise whitespace and numeric formatting. Round-trip each trip through the demo input reader and compare track values, node positions and arc IDs to the source. Stage the full partition before publishing its files; the inventory is written last. Generated outputs from an earlier source partition should be moved aside before processing a different set of trip IDs.

In [4]:
def export_trip(directory, stage):
    net = read_trip(directory)
    if (np.abs(net.track[:, :2]) > [180, 90]).any() or (np.abs(net.nodes) > [180, 90]).any():
        raise ValueError(f"{directory.name}: coordinates outside longitude/latitude bounds")
    target = stage / directory.name
    target.mkdir()
    for suffix, values, fmt in (("track", net.track, "%.17g"), ("nodes", net.nodes, "%.17g"),
                                ("arcs", net.arcs, "%d")):
        np.savetxt(target / f"{directory.name}.{suffix}", values, fmt=fmt)
    clean = read_trip(target)
    for field in ("track", "nodes", "arcs"):
        np.testing.assert_array_equal(getattr(clean, field), getattr(net, field))

    ids, x = features(clean, DEFAULT_K)
    if not np.isfinite(x).all():
        raise ValueError(f"{directory.name}: nonfinite candidate features")
    n, k = ids.shape
    obs = np.repeat(np.arange(n), k)
    arcs = ids.ravel()
    meta = pd.DataFrame({
        "observation": obs, "candidate_rank": np.tile(np.arange(1, k+1), n),
        "longitude": clean.track[obs, 0], "latitude": clean.track[obs, 1],
        "timestamp_s": clean.track[obs, 2], "candidate_arc_id": arcs,
        "from_node": clean.arcs[arcs, 0], "to_node": clean.arcs[arcs, 1]})
    table = pd.concat([meta, pd.DataFrame(x.reshape(-1, len(FEATURES)), columns=FEATURES)], axis=1)
    table.to_csv(target / "candidates.csv", index=False)
    dt = np.diff(clean.track[:, 2])
    return dict(trip=directory.name, observations=n, nodes=len(clean.nodes), arcs=len(clean.arcs),
                candidate_rows=len(table), candidates_per_observation=k,
                duplicate_times=int((dt == 0).sum()), max_gap_s=float(dt.max()) if len(dt) else 0.,
                zero_length_arcs=int((clean.length == 0).sum()),
                csv=f"{directory.name}/candidates.csv")

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT.exists():
    stale = {p.name for p in OUTPUT.iterdir() if p.is_dir()} - {p.name for p in trips}
    if stale:
        raise ValueError(f"Unexpected old trip directories in {OUTPUT}: {sorted(stale)}")
with TemporaryDirectory(dir=OUTPUT.parent) as staging:
    stage = Path(staging)
    inventory = pd.DataFrame([export_trip(directory, stage) for directory in trips])
    inventory.to_csv(stage / "inventory.csv", index=False)
    OUTPUT.mkdir(exist_ok=True)
    for directory in trips:
        target = OUTPUT / directory.name
        target.mkdir(exist_ok=True)
        for file in (stage / directory.name).iterdir():
            file.replace(target / file.name)
    (stage / "inventory.csv").replace(OUTPUT / "inventory.csv")

display(inventory)
display(inventory[["observations", "candidate_rows"]].sum().rename("partition total").to_frame())

,trip,observations,nodes,arcs,candidate_rows,candidates_per_observation,duplicate_times,max_gap_s,zero_length_arcs,csv
0,00000015,6854,1571,3225,54832,8,0,1.000,0,00000015/candidates.csv
1,00000017,1346,36724,76425,10768,8,0,8.000,2,00000017/candidates.csv
2,00000023,458,5053,10596,3664,8,0,1.000,0,00000023/candidates.csv
3,00000030,654,13223,24218,5232,8,0,14.000,0,00000030/candidates.csv
4,00000049,2642,23165,47542,21136,8,0,2.000,0,00000049/candidates.csv
5,00000059,1031,12631,24024,8248,8,0,4.000,0,00000059/candidates.csv
6,00000067,916,2265,4485,7328,8,0,19.000,0,00000067/candidates.csv
7,00000073,1144,12151,25631,9152,8,0,220.001,0,00000073/candidates.csv
8,00000087,1399,55234,107451,11192,8,0,2.000,3,00000087/candidates.csv
9,00000094,3230,6815,14931,25840,8,0,4.000,0,00000094/candidates.csv


,partition total
observations,19674
candidate_rows,157392


In [5]:
example_csv = OUTPUT / inventory.iloc[0]["csv"]
print(f"First observation from {example_csv}")
display(pd.read_csv(example_csv, nrows=int(inventory.iloc[0]["candidates_per_observation"])))

First observation from cleaned_data/evaluator_test/00000015/candidates.csv


,observation,candidate_rank,longitude,latitude,timestamp_s,candidate_arc_id,from_node,to_node,log_distance,distance_above_best,along_fraction,log_length,incoming_alignment,outgoing_alignment,window_alignment,incoming_speed,outgoing_speed,incoming_overshoot,outgoing_overshoot
0,0,1,4.753815,50.827698,1.0,1258,583,1470,1.573170,0.000000e+00,0.568723,4.215401,0.0,-0.910281,-0.923849,0.0,3.034904,0.0,0.000000
1,0,2,4.753815,50.827698,1.0,3035,1470,583,1.573170,1.332268e-15,0.431277,4.215401,0.0,0.910281,0.923849,0.0,3.034904,0.0,0.000000
2,0,3,4.753815,50.827698,1.0,1272,588,1470,3.402134,3.265996e+00,1.000000,2.175755,0.0,0.783180,0.803897,0.0,3.034904,0.0,3.034904
3,0,4,4.753815,50.827698,1.0,3036,1470,588,3.402134,3.265996e+00,0.000000,2.175755,0.0,-0.783180,-0.803897,0.0,3.034904,0.0,2.564192
4,0,5,4.753815,50.827698,1.0,1265,585,588,3.632263,3.525665e+00,1.000000,4.258209,0.0,0.291579,0.323979,0.0,3.034904,0.0,3.034904
5,0,6,4.753815,50.827698,1.0,1273,588,585,3.632263,3.525665e+00,0.000000,4.258209,0.0,-0.291579,-0.323979,0.0,3.034904,0.0,0.000000
6,0,7,4.753815,50.827698,1.0,1257,583,587,3.667092,3.564336e+00,0.000000,2.743046,0.0,0.966624,0.974787,0.0,3.034904,0.0,1.834939
7,0,8,4.753815,50.827698,1.0,1259,583,580,3.667092,3.564336e+00,0.000000,3.444078,0.0,-0.138804,-0.105004,0.0,3.034904,0.0,0.000000


## Run the demo

From the project root, use `./run_local.sh --evaluator` on one laptop or `./run_local.sh --lan --evaluator` on Laptop 1. Click **Load trips** and select an evaluator trip. The live predictor reads the prepared track and graph and computes features for the frozen classifier on each request. The CSVs are the inspectable export of that same transformation.